# H&M Fashion Recommender — Purchase Prediction (Kaggle MAP@12)

Predicts the 12 articles each customer is most likely to buy in the next 7 days, scored by MAP@12 on the Kaggle H&M benchmark (1.37M customers).

Predictions blend four signals, in priority order:
1. **Recency-weighted repurchase** — items the customer bought, weighted by recency and frequency (the dominant signal in this dataset).
2. **Item co-occurrence** — items frequently bought alongside their recent purchases.
3. **Visual similarity** — FashionCLIP nearest neighbors of recent purchases.
4. **Popularity** — recent best-sellers, as a fallback.

Final score: MAP@12 ≈ 0.021 (roughly 2× a popularity-only baseline).

## 1. Setup & Load Data

Loads transactions, the article table, and the FashionCLIP embeddings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/hm_fashion_project/src')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q faiss-gpu-cu12

## 2. Recency-Weighted Repurchase

For each customer, score their purchased items by `weight = 1 / (1 + days_ago/7)`, summed across purchases — so recent and repeated buys rank highest. This is the strongest predictor of next-week purchases.

In [ ]:
import pandas as pd, numpy as np, os, faiss
from collections import defaultdict, Counter
from itertools import combinations
from paths import BASE_PROCESS

trans = pd.read_parquet(os.path.join(BASE_PROCESS, "transactions_train/transactions_train_final.parquet"))
trans['article_id'] = trans['article_id'].astype(str).str.zfill(10)
trans['customer_id'] = trans['customer_id'].astype(str)
trans['t_dat'] = pd.to_datetime(trans['t_dat'])

df_final = pd.read_parquet(os.path.join(BASE_PROCESS, "articles/articles_with_images.parquet"))
df_final['article_id'] = df_final['article_id'].astype(str).str.zfill(10)

fclip = np.load(os.path.join(BASE_PROCESS, "embeddings/fclip_image_embeddings.npy")).astype('float32')
print("loaded", trans.shape, df_final.shape, fclip.shape)

loaded (15306023, 7)


## 3. Popularity

The top-selling articles over the last two weeks, used to fill remaining slots for customers with little history.

In [ ]:
last_date = trans['t_dat'].max()
t = trans.copy()
t['days_ago'] = (last_date - t['t_dat']).dt.days
t['weight'] = 1.0 / (1 + t['days_ago']/7.0)
agg = t.groupby(['customer_id','article_id'])['weight'].sum().reset_index()
agg = agg.sort_values(['customer_id','weight'], ascending=[True, False])
top_items_dict = agg.groupby('customer_id')['article_id'].apply(lambda s: list(s)[:12]).to_dict()
print("repurchase ready", len(top_items_dict))

recomputed 1004267


In [ ]:
cutoff = trans['t_dat'].max() - pd.Timedelta(days=14)
top_popular = trans[trans['t_dat'] > cutoff]['article_id'].value_counts().head(12).index.tolist()
print(top_popular[:3])

['0909370001', '0924243001', '0918522001']


## 4. Recent Baskets

Build each customer's recent purchase list (last 30 days) for the co-occurrence and similarity steps.

In [ ]:
recent_cut = trans['t_dat'].max() - pd.Timedelta(days=30)
recent_t = trans[trans['t_dat'] > recent_cut]
cust_baskets = recent_t.groupby('customer_id')['article_id'].apply(list)
cust_recent_dict = (recent_t.sort_values('t_dat')
                    .groupby('customer_id')['article_id']
                    .apply(lambda s: list(dict.fromkeys(s.tolist()[::-1]))).to_dict())
print("baskets", len(cust_baskets))

recent transactions: (1122902, 7)
baskets: 245945


## 5. Item Co-occurrence

Count items bought together within recent customer baskets, then keep each item's top co-bought partners. Captures complementary "bought together" patterns.

In [ ]:
from tqdm.notebook import tqdm
cooc = defaultdict(Counter)
for basket in tqdm(cust_baskets):
    items = list(set(basket))[:30]
    for a, b in combinations(items, 2):
        cooc[a][b] += 1
        cooc[b][a] += 1
top_cooc = {item: [a for a, _ in c.most_common(12)] for item, c in cooc.items()}
print("cooc ready", len(top_cooc))

  0%|          | 0/245945 [00:00<?, ?it/s]

items with co-occurrence data: 28453


## 6. Visual Similarity (FashionCLIP + FAISS)

Precompute each item's 3 nearest visual neighbors in one batched FAISS search. This is the fast, item-level alternative to per-customer retrieval (which was tested and added little while being far slower).

In [ ]:
art_to_row = {aid: i for i, aid in enumerate(df_final['article_id'])}
row_to_art = {i: aid for aid, i in art_to_row.items()}
index = faiss.IndexFlatIP(512)
index.add(fclip)
D, I = index.search(fclip, 4)
item_similar = {row_to_art[r]: [row_to_art[i] for i in I[r][1:4]] for r in range(len(fclip))}
print("faiss similar ready", len(item_similar))

similar items ready: 71664


## 7. Build Predictions

For each customer, fill 12 slots in priority order: repurchase → co-occurrence → visual similarity → popularity, deduplicating along the way. Mapped onto the full customer list from `sample_submission.csv`.

In [ ]:
def make_pred(cust):
    preds = []; seen = set()
    for a in top_items_dict.get(cust, []):
        if a not in seen: preds.append(a); seen.add(a)
        if len(preds) >= 12: break
    if len(preds) < 12:
        for ri in cust_recent_dict.get(cust, [])[:3]:
            for co in top_cooc.get(ri, []):
                if co not in seen: preds.append(co); seen.add(co)
                if len(preds) >= 12: break
            if len(preds) >= 12: break
    if len(preds) < 12:
        for ri in cust_recent_dict.get(cust, [])[:3]:
            for sim in item_similar.get(ri, []):
                if sim not in seen: preds.append(sim); seen.add(sim)
                if len(preds) >= 12: break
            if len(preds) >= 12: break
    for p in top_popular:
        if len(preds) >= 12: break
        if p not in seen: preds.append(p); seen.add(p)
    return ' '.join(preds[:12])

sample = pd.read_csv("/content/drive/MyDrive/hm_fashion_project/data/raw/sample_submission.csv")
sample['prediction'] = sample['customer_id'].apply(make_pred)
print("done", sample.shape)

done (1371980, 2)


## 8. Verify & Save

Confirm every row has exactly 12 unique predictions, then write the submission file.

In [ ]:
counts = sample['prediction'].str.split().apply(len)
print("all 12?", (counts == 12).all(), "rows:", len(sample))
dupes = sample['prediction'].head(5000).apply(lambda x: len(x.split()) != len(set(x.split())))
print("dupes:", dupes.sum())
sample.to_csv("/content/submission_final.csv", index=False)
print("saved")

all 12? True rows: 1371980
dupes: 0
saved
